In [ ]:
import cv2
import numpy as np
import pandas as pd
import os
from tqdm import tqdm
from skimage.feature import graycomatrix, graycoprops 

def extract_features(image_path, label):
    img_bgr = cv2.imread(image_path)
    if img_bgr is None:
        return None
        
    # --- MASKING SEL ---
    img_gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    img_blur = cv2.GaussianBlur(img_gray, (5, 5), 0)
    _, thresh_cell = cv2.threshold(img_blur, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
    mask_cell = cv2.bitwise_not(thresh_cell) 

    # --- MASKING PARASIT (S-Channel) ---
    img_hsv = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2HSV)
    h, s, v = cv2.split(img_hsv) 
    s_inside_cell = s[mask_cell == 255]
    
    if len(s_inside_cell) == 0:
        return {'cell_area': 0, 'parasite_count': 0, 'parasite_area': 0, 'texture_contrast': 0, 'label': label}
        
    mean_s = np.mean(s_inside_cell)
    std_s = np.std(s_inside_cell)
    batas_parasit = mean_s + (3 * std_s) + 10
    
    _, mask_parasite_raw = cv2.threshold(s, batas_parasit, 255, cv2.THRESH_BINARY)
    mask_parasite_final = cv2.bitwise_and(mask_parasite_raw, mask_cell)
    
    # --- EKSTRAKSI FITUR BENTUK (Contours) ---
    contours_cell, _ = cv2.findContours(mask_cell, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    contours_parasite, _ = cv2.findContours(mask_parasite_final, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    cell_area = 0
    # Cari kontur sel terbesar untuk menghitung tekstur (biar debu di luar sel tidak ikut terhitung)
    largest_cell_cnt = None 
    max_area = 0
    
    for cnt in contours_cell:
        area = cv2.contourArea(cnt)
        if area > 500:
            cell_area += area
            if area > max_area:
                max_area = area
                largest_cell_cnt = cnt
                
    parasite_area = sum([cv2.contourArea(cnt) for cnt in contours_parasite if cv2.contourArea(cnt) > 25])
    parasite_count = sum([1 for cnt in contours_parasite if cv2.contourArea(cnt) > 25])

    # --- EKSTRAKSI FITUR TEKSTUR (GLCM) ---
    texture_contrast = 0
    if largest_cell_cnt is not None:
        # Buat "kotak pembungkus" (Bounding Box) di sekitar sel darah
        x, y, w, h = cv2.boundingRect(largest_cell_cnt)
        
        # Potong gambar abu-abu HANYA sejumput kotak sel itu saja
        cell_crop = img_gray[y:y+h, x:x+w]
        
        # GLCM menghitung seberapa drastis perubahan piksel tetangga (jarak 1 piksel, sudut 0 derajat horizontal)
        glcm = graycomatrix(cell_crop, distances=[1], angles=[0], levels=256, symmetric=True, normed=True)
        
        # Ambil nilai 'contrast' (semakin tinggi angkanya = semakin kasar/banyak bercak)
        texture_contrast = graycoprops(glcm, 'contrast')[0, 0]

    return {
        'cell_area': cell_area,
        'parasite_count': parasite_count,
        'parasite_area': parasite_area,
        'texture_contrast': texture_contrast, 
        'label': label 
    }

# --- JALANIN LOOPING KE SELURUH FOLDER ---
dataset_dir = '../data/' 
categories = ['Parasitized', 'Uninfected']
all_extracted_data = []

print("Memulai proses ekstraksi fitur. Silakan tunggu, ini bisa memakan waktu beberapa menit...")

for category in categories:
    folder_path = os.path.join(dataset_dir, category)
    # Label: Sakit (Parasitized) = 1, Sehat (Uninfected) = 0
    label = 1 if category == 'Parasitized' else 0
    
    # Ambil semua nama file berakhiran .png di dalam folder
    image_files = [f for f in os.listdir(folder_path) if f.endswith('.png')]
    
    # Looping dengan progress bar (tqdm)
    for file in tqdm(image_files, desc=f"Memproses {category}"):
        img_path = os.path.join(folder_path, file)
        
        # Panggil fungsi yang kita buat di atas
        features = extract_features(img_path, label)
        
        if features is not None:
            all_extracted_data.append(features)

# --- SIMPAN KE FORMAT CSV ---
# Ubah list of dictionary menjadi Pandas DataFrame
df = pd.DataFrame(all_extracted_data)

# Simpan jadi file CSV
output_filename = 'malaria_features.csv'
df.to_csv(output_filename, index=False)

print(f"\nSelesai! Data berhasil disimpan ke {output_filename}")
print("Contoh 5 data pertama:")
print(df.head())

Memulai proses ekstraksi fitur. Silakan tunggu, ini bisa memakan waktu beberapa menit...


Memproses Uninfected: 100%|██████████| 13679/13679 [00:34<00:00, 395.00it/s]



Selesai! Data berhasil disimpan ke malaria_features.csv
Contoh 5 data pertama:
   cell_area  parasite_count  parasite_area  texture_contrast  label
0    16954.0               1          208.5        366.991694      1
1    11529.5               1          144.0        445.258874      1
2    11486.0               1          215.0        389.118989      1
3    18251.0               2          172.5        389.324830      1
4    14787.0               1          110.0        440.254214      1
